In [63]:
import numpy as np
import pandas as pd 
import plotly.express as px 
import matplotlib.pyplot as plt
import pymap3d as p3d 

In [97]:
right_pts = pd.read_csv('right_interpolated.csv',names=['lat', 'lon', 'alt'])
left_pts = pd.read_csv('left_interpolated.csv', names=['lat','lon','alt'])
cent_pts = pd.read_csv('cerpm_waypoints.csv',names=['lat','lon','alt','heading'])

resampled_df = pd.read_csv('resampled_pts1.csv').drop('Unnamed: 0', axis=1)

cent_pts = cent_pts.loc[::-1].reset_index(drop=True)
right_pts = right_pts.loc[::-1].reset_index(drop=True)
left_pts = left_pts.loc[::-1].reset_index(drop=True)


lat0 = cent_pts.loc[0]['lat']
lon0 = cent_pts.loc[0]['lon']
alt0 = cent_pts.loc[0]['alt']

distance = 3.38/2.0 # meters


In [105]:
idx = 80

right_x, right_y, right_z = p3d.geodetic2ned(right_pts.loc[idx]['lat'], right_pts.loc[idx]['lon'], right_pts.loc[idx]['alt'], lat0, lon0, alt0)
left_x, left_y, left_z = p3d.geodetic2ned(left_pts.loc[idx]['lat'], left_pts.loc[idx]['lon'], left_pts.loc[idx]['alt'], lat0, lon0, alt0)

x_local, y_local, z_local = p3d.geodetic2ned(cent_pts.loc[idx]['lat'], cent_pts.loc[idx]['lon'], cent_pts.loc[idx]['alt'], lat0, lon0, alt0)

offset_x = y_local - left_y
offset_y = left_x - x_local 

yaw = np.arccos(offset_x / distance)



In [106]:
yaw

1.327100613550206

In [107]:
def calculate_yaw(left_df, cent_df, right_df, distance, lat0, lon0, alt0):
    yaw_angles = []
    for idx in left_df.index:
        print(idx)
        right_x, right_y, right_z = p3d.geodetic2ned(right_df.loc[idx]['lat'], right_df.loc[idx]['lon'], right_df.loc[idx]['alt'], lat0, lon0, alt0)
        left_x, left_y, left_z = p3d.geodetic2ned(left_df.loc[idx]['lat'], left_df.loc[idx]['lon'], left_df.loc[idx]['alt'], lat0, lon0, alt0)
        x_local, y_local, z_local = p3d.geodetic2ned(cent_df.loc[idx]['lat'], cent_df.loc[idx]['lon'], cent_df.loc[idx]['alt'], lat0, lon0, alt0)
        
        offset_x = y_local - left_y
        offset_y = left_x - x_local 
        
        yaw = np.arccos(offset_x / distance)
        yaw_angles.append(yaw)
    
    return yaw_angles


In [109]:
distance = 3.38/2
yaw_angles = calculate_yaw(left_pts, cent_pts, right_pts, distance, lat0, lon0, alt0)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99


/tmp/ipykernel_112503/4121006220.py:12: RuntimeWarning:

invalid value encountered in arccos



In [87]:
fig = px.scatter_mapbox(right_pts, lat="lat", lon="lon", 
                        zoom=16, mapbox_style="open-street-map",
                        width=900,height=500)

fig.add_scattermapbox(lat=left_pts['lat'], lon=left_pts['lon'], mode='markers', name='Right Boundary')
fig.add_scattermapbox(lat=cent_pts['lat'], lon=cent_pts['lon'], mode='markers', name='Right Boundary')

# fig.add_scattermapbox(lat=[right_pts.iloc[0]['lat']], lon=[right_pts.iloc[0]['lon']], mode='markers', name='Right Boundary')

# fig.add_scattermapbox(lat=[cent_pts.iloc[-2]['lat']], lon=[cent_pts.iloc[-2]['lon']], mode='markers', name='left Boundary')

fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()